### Tools 

Model can request to call tools to perform tasks such as fetching data from a database ,  searching the web , or running code . Tools are pairings 

1. A schema , including the name of the tool , a description , and/or argument definitions (often a JSON schema)

2. A function or coroutine excecute


In [1]:
import os 
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-120b")
res = model.invoke("Why do parrots talk")
res


AIMessage(content='Parrots aren’t “talking” in the human sense— they are **vocal mimics** that have evolved a set of traits that make them exceptionally good at copying sounds they hear. Here’s why they can produce words and phrases:\n\n| Reason | How it works | Why it matters |\n|--------|--------------|----------------|\n| **Highly developed vocal organ (the syrinx)** | The syrinx, located at the base of a bird’s trachea, contains many finely tuned muscles that allow precise control of pitch, tone, and rhythm. Parrots have one of the most flexible syrinxes of any bird. | This mechanical flexibility lets them reproduce a wide range of sounds, from simple chirps to complex human speech. |\n| **Large, socially‑oriented brain** | Parrots have a proportionally large forebrain (especially the nidopallium) that supports learning, memory, and social cognition. | Social intelligence drives the need to recognize and imitate the calls of flock mates, which in captivity often means the human car

In [2]:
from langchain.tools import tool

@tool
def get_weather(location:str)-> str:
    """Get the weather of the location"""
    return f"its sunny in {location}"

model_with_tools = model.bind_tools([get_weather])

In [3]:
res = model_with_tools.invoke("Whats the weather in delhi")
print(res)

for tool_call in res.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'The user asks: "Whats the weather in delhi". We need to get weather via function get_weather. Use function.', 'tool_calls': [{'id': 'fc_ec23def8-ad12-4545-ba26-96beb74d5d2f', 'function': {'arguments': '{"location":"Delhi"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 126, 'total_tokens': 179, 'completion_time': 0.11163471, 'completion_tokens_details': {'reasoning_tokens': 26}, 'prompt_time': 0.005430314, 'prompt_tokens_details': None, 'queue_time': 0.406489732, 'total_time': 0.117065024}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_93703442d9', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a07fdf-9dfb-7760-9c84-25b46f5271fa-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Delhi'}, 'id': 'fc_ec23def8-ad12-4545-ba26-96beb74d5d2f', 'type': 'tool_call'}] inva

### Tool Excecution Loop

In [4]:
## Step - 1 Model generates tool calls 
message = [{"role": "user" , "content": "What is the weather in delhi"}]
ai_msg = model_with_tools.invoke(message)
message.append(ai_msg)

## Step 2 -  Excecute the tools and collect result 

for tool_call in ai_msg.tool_calls:
    ## Excecute the call with the generated argument
    tool_result = get_weather.invoke(tool_call)
    message.append(tool_result)

## Step - 3 : Pass result back to model for final answer

final_res = model_with_tools.invoke(message)
print(final_res.text)

The current weather in Delhi is sunny. Enjoy the clear skies!


In [5]:
message

[{'role': 'user', 'content': 'What is the weather in delhi'},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks: "What is the weather in delhi". We need to get weather via function get_weather. Use location "Delhi".', 'tool_calls': [{'id': 'fc_ff2d39f7-f7e7-41ca-b0e3-84176a59f741', 'function': {'arguments': '{"location":"Delhi"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 55, 'prompt_tokens': 127, 'total_tokens': 182, 'completion_time': 0.116743501, 'completion_tokens_details': {'reasoning_tokens': 28}, 'prompt_time': 0.01488428, 'prompt_tokens_details': None, 'queue_time': 0.354306646, 'total_time': 0.131627781}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_dee443f41b', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a07fe4-3b27-79c3-ad75-d3615375ff0c-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'D